## Getting Started with LangChain and Foundry Agents

### Installing Utilities and Libraries

In [ ]:
%pip install langchain-azure-ai==1.2.10 azure-ai-projects==2.7.0 azure-identity==1.25.3 azure-ai-documentintelligence==1.0.2

### Setting up the Environment Variables

In [ ]:
import os
from dotenv import load_dotenv
from azure.identity import AzureCliCredential
from langchain_azure_ai.utils.agents import pretty_print

load_dotenv()
foundry_project_endpoint = os.getenv("FOUNDRY_PROJECT_ENDPOINT")
model_deployment_name = os.getenv("MODEL_DEPLOYMENT_NAME")

credential = AzureCliCredential()

### Create the Agent Service Factory Client

In [ ]:
from langchain_core.messages import HumanMessage
from langchain_azure_ai.agents import AgentServiceFactory

factory = AgentServiceFactory(
	project_endpoint=foundry_project_endpoint,
	credential=credential,
)	

### Create a Basic Prompt Agent

In [ ]:
agent = factory.create_prompt_agent(
	name="my-echo-agent",
	model=model_deployment_name,
	instructions=(
		"You are a helpful AI assistant that always replies with the "
		"opposite of what the user says."
	),
)

agent_ids = factory.get_agents_id_from_graph(agent)
print(f"Agent created: {next(iter(agent_ids))}")

### Query your Agent

In [ ]:
messages = [HumanMessage(content="I'm a genius and I love programming!")]
response = agent.invoke({"messages": messages})
pretty_print(response)

### Create a Document Intelligence Agent

In [ ]:
from langchain_azure_ai.tools import AzureAIDocumentIntelligenceTool

document_parser_agent = factory.create_prompt_agent(
	name="document-agent",
	model=model_deployment_name,
	instructions="You are a helpful assistant tasked with analyzing documents.",
	tools=[AzureAIDocumentIntelligenceTool()],
)

### Ask the Agent to Analyze an Invoice using the Invoice URL

In [ ]:
messages = [
	HumanMessage(
		content=(
			"What's the total amount in the invoice at "
			"https://raw.githubusercontent.com/Azure/azure-sdk-for-python/main/"
			"sdk/formrecognizer/azure-ai-formrecognizer/tests/sample_forms/"
			"forms/Form_1.jpg"
		)
	)
]
response = document_parser_agent.invoke({"messages": messages})
pretty_print(response)